# Notebook 6 — Tableau Visualization Export
## HMDA 2023 Loan Denial Prediction

**Objective:** Export CSV files for four Tableau dashboards.
---

In [7]:
# ============================================================
# Imports & SparkSession
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import pandas as pd
import json, os, warnings
warnings.filterwarnings('ignore')

spark = (SparkSession.builder
    .appName("HMDA_2023_TableauExport_6")
    .master("local[2]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

DATA_DIR = "../data/processed"
TABLEAU_DIR = "../tableau/data"
os.makedirs(TABLEAU_DIR, exist_ok=True)

# Load processed Parquet
df = spark.read.parquet(f"{DATA_DIR}/hmda_2023.parquet")
df_target = df.filter(F.col('action_taken').isin(1, 3)).withColumn(
    'label', F.when(F.col('action_taken') == 3, 1).otherwise(0)
)

# Load model results
with open(f"{DATA_DIR}/model_results_4.json") as f:
    ALL_RESULTS = json.load(f)

# Load evaluation report if available
try:
    with open(f"{DATA_DIR}/results/final_evaluation_report.json") as f:
        eval_report = json.load(f)
except FileNotFoundError:
    eval_report = {}

total_rows = df.count()
target_rows = df_target.count()
print(f"Full dataset: {total_rows:,} rows")
print(f"Target population: {target_rows:,} rows")

26/03/02 02:32:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/02 02:32:10 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/02 02:32:10 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Full dataset: 11,483,889 rows
Target population: 7,686,484 rows


## Dashboard 1: Data Quality & Loan Patterns

In [8]:
# ============================================================
# Dashboard 1: Data Quality & Loan Patterns
# ============================================================

dq_frames = []

# 1a. KPI Summary
kpi = df_target.agg(
    F.count('*').alias('total_records'),
    F.sum('label').alias('total_denied'),
    F.avg('label').alias('denial_rate'),
    F.avg('loan_amount').alias('avg_loan_amount'),
).toPandas()
kpi['data_type'] = 'kpi_summary'
dq_frames.append(kpi)

# 1b. Action taken distribution (all actions)
action_dist = df.groupBy('action_taken').count().orderBy('action_taken').toPandas()
action_dist['data_type'] = 'action_distribution'
dq_frames.append(action_dist)

# 1c. Missing values by column (top 30)
null_data = []
for col_name in df.columns:
    null_count = df.filter(F.col(col_name).isNull() | F.col(col_name).isin('Exempt', 'NA', '')).count()
    null_data.append({'column': col_name, 'null_count': null_count,
                      'null_pct': round(null_count / total_rows * 100, 2)})
null_df = pd.DataFrame(null_data).nlargest(30, 'null_pct')
null_df['data_type'] = 'missing_values'
dq_frames.append(null_df)

# 1d. Loan amount distribution by action
loan_dist = (df_target
    .withColumn('loan_bucket', F.when(F.col('loan_amount') < 100000, '<100K')
        .when(F.col('loan_amount') < 250000, '100K-250K')
        .when(F.col('loan_amount') < 500000, '250K-500K')
        .when(F.col('loan_amount') < 1000000, '500K-1M')
        .otherwise('1M+'))
    .groupBy('loan_bucket', 'label').count()
    .orderBy('loan_bucket', 'label').toPandas()
)
loan_dist['data_type'] = 'loan_distribution'
dq_frames.append(loan_dist)

# 1e. State-level denial rates
state_rates = (df_target
    .filter(F.col('state_code').isNotNull())
    .groupBy('state_code')
    .agg(F.count('*').alias('total'), F.avg('label').alias('denial_rate'))
    .orderBy(F.desc('total')).toPandas()
)
state_rates['data_type'] = 'state_denial_rates'
dq_frames.append(state_rates)

# Combine and save
dq_all = pd.concat(dq_frames, ignore_index=True)
dq_path = f"{TABLEAU_DIR}/dashboard_1_data_quality.csv"
dq_all.to_csv(dq_path, index=False)
# print(f"Dashboard 1 saved: {dq_path} ({len(dq_all)} rows)")

## Dashboard 2: Model Performance Comparison

In [9]:
# ============================================================
# Dashboard 2: Model Performance Comparison
# ============================================================

mp_frames = []

# 2a. Model metrics comparison
metrics_rows = []
for name, m in ALL_RESULTS.items():
    metrics_rows.append({
        'model': name,
        'ROC_AUC': m.get('ROC-AUC', 0),
        'PR_AUC': m.get('PR-AUC', 0),
        'F1': m.get('F1', 0),
        'Accuracy': m.get('Accuracy', 0),
        'Denial_Precision': m.get('Denial_Precision', 0),
        'Denial_Recall': m.get('Denial_Recall', 0),
        'Denial_F1': m.get('Denial_F1', 0),
        'Train_Time_s': m.get('Train_Time_s', 0),
    })
metrics_df = pd.DataFrame(metrics_rows)
metrics_df['data_type'] = 'model_metrics'
mp_frames.append(metrics_df)

# 2b. Bootstrap CIs
bootstrap_ci = eval_report.get('bootstrap_confidence_intervals', {})
ci_rows = []
for name, ci in bootstrap_ci.items():
    ci_rows.append({
        'model': name,
        'pr_auc_mean': ci.get('pr_auc_mean', 0),
        'pr_auc_ci_lower': ci.get('pr_auc_ci_lower', 0),
        'pr_auc_ci_upper': ci.get('pr_auc_ci_upper', 0),
        'roc_auc_mean': ci.get('roc_auc_mean', 0),
        'roc_auc_ci_lower': ci.get('roc_auc_ci_lower', 0),
        'roc_auc_ci_upper': ci.get('roc_auc_ci_upper', 0),
    })
if ci_rows:
    ci_df = pd.DataFrame(ci_rows)
    ci_df['data_type'] = 'bootstrap_ci'
    mp_frames.append(ci_df)

# 2c. Confusion matrices
cm_rows = []
for name, m in ALL_RESULTS.items():
    cm = m.get('Confusion', {})
    cm_rows.append({
        'model': name, 'TP': cm.get('TP', 0), 'FP': cm.get('FP', 0),
        'FN': cm.get('FN', 0), 'TN': cm.get('TN', 0)
    })
cm_df = pd.DataFrame(cm_rows)
cm_df['data_type'] = 'confusion_matrix'
mp_frames.append(cm_df)

mp_all = pd.concat(mp_frames, ignore_index=True)
mp_path = f"{TABLEAU_DIR}/dashboard_2_model_performance.csv"
mp_all.to_csv(mp_path, index=False)
# print(f"Dashboard 2 saved: {mp_path} ({len(mp_all)} rows)")

## Dashboard 3: Fair Lending & Business Insights

In [10]:
# ============================================================
# Dashboard 3: Fair Lending & Business Insights
# ============================================================

fl_frames = []

# 3a. Denial rates by race
race_rates = (df_target
    .groupBy('derived_race')
    .agg(F.count('*').alias('total'), F.avg('label').alias('denial_rate'),
         F.avg('loan_amount').alias('avg_loan_amount'))
    .orderBy(F.desc('total')).toPandas()
)
race_rates['data_type'] = 'denial_by_race'
fl_frames.append(race_rates)

# 3b. Denial rates by ethnicity
eth_rates = (df_target
    .groupBy('derived_ethnicity')
    .agg(F.count('*').alias('total'), F.avg('label').alias('denial_rate'),
         F.avg('loan_amount').alias('avg_loan_amount'))
    .orderBy(F.desc('total')).toPandas()
)
eth_rates['data_type'] = 'denial_by_ethnicity'
fl_frames.append(eth_rates)

# 3c. Denial rates by sex
sex_rates = (df_target
    .groupBy('derived_sex')
    .agg(F.count('*').alias('total'), F.avg('label').alias('denial_rate'),
         F.avg('loan_amount').alias('avg_loan_amount'))
    .orderBy(F.desc('total')).toPandas()
)
sex_rates['data_type'] = 'denial_by_sex'
fl_frames.append(sex_rates)

# 3d. Denial by loan purpose
purpose_rates = (df_target
    .groupBy('loan_purpose')
    .agg(F.count('*').alias('total'), F.avg('label').alias('denial_rate'))
    .orderBy(F.desc('total')).toPandas()
)
purpose_rates['data_type'] = 'denial_by_purpose'
fl_frames.append(purpose_rates)

# 3e. Denial by loan type
type_rates = (df_target
    .groupBy('loan_type')
    .agg(F.count('*').alias('total'), F.avg('label').alias('denial_rate'))
    .orderBy(F.desc('total')).toPandas()
)
type_rates['data_type'] = 'denial_by_type'
fl_frames.append(type_rates)

# 3f. Denial by occupancy type
occ_rates = (df_target
    .groupBy('occupancy_type')
    .agg(F.count('*').alias('total'), F.avg('label').alias('denial_rate'))
    .orderBy(F.desc('total')).toPandas()
)
occ_rates['data_type'] = 'denial_by_occupancy'
fl_frames.append(occ_rates)

fl_all = pd.concat(fl_frames, ignore_index=True)
fl_path = f"{TABLEAU_DIR}/dashboard_3_fair_lending.csv"
fl_all.to_csv(fl_path, index=False)
# print(f"Dashboard 3 saved: {fl_path} ({len(fl_all)} rows)")

## Dashboard 4: Scalability & Cost Analysis

In [11]:
# ============================================================
# Dashboard 4: Scalability & Cost Analysis
# ============================================================

sc_frames = []

# 4a. Strong scaling
strong = eval_report.get('strong_scaling', [])
if strong:
    strong_df = pd.DataFrame(strong)
    strong_df['data_type'] = 'strong_scaling'
    sc_frames.append(strong_df)

# 4b. Weak scaling
weak = eval_report.get('weak_scaling', [])
if weak:
    weak_df = pd.DataFrame(weak)
    weak_df['data_type'] = 'weak_scaling'
    sc_frames.append(weak_df)

# 4c. Training time comparison
time_rows = []
for name, m in ALL_RESULTS.items():
    time_rows.append({
        'model': name,
        'train_time_s': m.get('Train_Time_s', 0),
        'cost_estimate': (m.get('Train_Time_s', 0) / 3600) * 0.50
    })
time_df = pd.DataFrame(time_rows)
time_df['data_type'] = 'training_time'
sc_frames.append(time_df)

# 4d. Training costs
costs = eval_report.get('training_costs', [])
if costs:
    cost_df = pd.DataFrame(costs)
    cost_df['data_type'] = 'training_cost'
    sc_frames.append(cost_df)

sc_all = pd.concat(sc_frames, ignore_index=True)
sc_path = f"{TABLEAU_DIR}/dashboard_4_scalability.csv"
sc_all.to_csv(sc_path, index=False)
# print(f"Dashboard 4 saved: {sc_path} ({len(sc_all)} rows)")

In [12]:
# ============================================================
# Summary & Cleanup
# ============================================================

print("=" * 70)
print("TABLEAU EXPORT SUMMARY")
print("=" * 70)

for f in os.listdir(TABLEAU_DIR):
    if f.endswith('.csv'):
        path = os.path.join(TABLEAU_DIR, f)
        size = os.path.getsize(path)
        print(f"  {f:<45} {size:>10,} bytes")

spark.stop()

TABLEAU EXPORT SUMMARY
  dashboard_1_data_quality.csv                       6,171 bytes
  dashboard_4_scalability.csv                        1,129 bytes
  dashboard_2_model_performance.csv                  1,801 bytes
  dashboard_3_fair_lending.csv                       2,332 bytes
